In [254]:
def line():
    print('-'*50)
    print('\n')

In [255]:
import  pandas as pd
import cv2


df = pd.read_csv("material.csv")


print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 16555 entries, 0 to 16554
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   SID            16555 non-null  int64
 1   DOI            16555 non-null  str  
 2   composition    16472 non-null  str  
 3   sample_id      16555 non-null  int64
 4   figure_id      16555 non-null  int64
 5   figure_name    16555 non-null  str  
 6   prop_x         16555 non-null  str  
 7   prop_y         16554 non-null  str  
 8   unit_x         16555 non-null  str  
 9   unit_y         16555 non-null  str  
 10  x              16555 non-null  str  
 11  y              16555 non-null  str  
 12  created_at     16555 non-null  str  
 13  updated_at     16555 non-null  str  
 14  project_names  16555 non-null  str  
 15  comments       16451 non-null  str  
dtypes: int64(3), str(13)
memory usage: 2.0 MB
None


In [256]:
# duplicet in each columns
for col in df.columns:
    print(f"{col}: {df[col].duplicated().sum()} duplicates")

SID: 14347 duplicates
DOI: 14347 duplicates
composition: 9512 duplicates
sample_id: 6651 duplicates
figure_id: 11912 duplicates
figure_name: 15650 duplicates
prop_x: 16548 duplicates
prop_y: 16539 duplicates
unit_x: 16548 duplicates
unit_y: 16545 duplicates
x: 660 duplicates
y: 603 duplicates
created_at: 6 duplicates
updated_at: 6 duplicates
project_names: 16546 duplicates
comments: 15561 duplicates


In [257]:
df = df.loc[
    (df['prop_x'] == 'Dielectric loss (tanδ)') |
    (df['prop_y'] == 'Dielectric loss (tanδ)')
]

print(df.shape)

(5298, 16)


In [258]:
#unique data in each columns

print("prop_x:")
for value in df['prop_x'].dropna().unique():
    print(value)

line()
print("\nprop_y:")
for value in df['prop_y'].dropna().unique():
    print(value)

line()




prop_x:
Temperature
--------------------------------------------------



prop_y:
Dielectric loss (tanδ)
--------------------------------------------------




In [259]:
import pandas as pd
import ast

# Convert string representations of lists into actual lists
df['Temperature(K)'] = df['x'].apply(ast.literal_eval)
df['Dielectric loss (tanδ)'] = df['y'].apply(ast.literal_eval)

# Delete unnecessary columns
df.drop(
    columns=['prop_x', 'prop_y', 'unit_x', 'unit_y', 'x', 'y','created_at','updated_at','project_names','figure_name','figure_id','comments'],
    inplace=True
)

df.drop(
    df[
        df['Temperature(K)'].astype(str).str.contains('[]', na=False, regex=False) |
        df['composition'].isna() |
        df['composition'].astype(str).str.strip().isin(['', '<null>', 'null', 'None'])|
        df['Dielectric loss (tanδ)'].astype(str).str.contains('[]', na=False, regex=False) |
        df['composition'].str.contains('-', na=False, regex=False)

    ].index,
    inplace=True
)

print(df.head())

       SID                   DOI        composition  sample_id  \
129  51340    10.1111/jace.16359    Ba4La2Sn4Nb6O30      93953   
131  51340    10.1111/jace.16359    Ba4Nd2Sn4Nb6O30      93954   
137  51345  10.1143/jjap.42.5913  Ba0.66Ho0.22Nb2O6      93958   
139  51345  10.1143/jjap.42.5913  Ba0.66Er0.22Nb2O6      93959   
141  51340    10.1111/jace.16359    Ba4Sm2Sn4Nb6O30      93960   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [260]:
print(df)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [261]:
# Count NaN values in each column
for col in df.columns:
    print(f"{col}: {df[col].isna().sum()} missing values")


print(df.info())
line()
print(df.shape)


SID: 0 missing values
DOI: 0 missing values
composition: 0 missing values
sample_id: 0 missing values
Temperature(K): 0 missing values
Dielectric loss (tanδ): 0 missing values
<class 'pandas.DataFrame'>
Index: 5173 entries, 129 to 16420
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   SID                     5173 non-null   int64 
 1   DOI                     5173 non-null   str   
 2   composition             5173 non-null   str   
 3   sample_id               5173 non-null   int64 
 4   Temperature(K)          5173 non-null   object
 5   Dielectric loss (tanδ)  5173 non-null   object
dtypes: int64(2), object(2), str(2)
memory usage: 282.9+ KB
None
--------------------------------------------------


(5173, 6)


In [262]:
df.to_csv("updated_data.csv", index=False)
print("file updated")

file updated


Finding the "AND" compostion

In [263]:
import re
import cv2
import pandas as pd

df = pd.read_csv("updated_data.csv")


ALLOWED_COMPOUNDS = { "CaO","K2O","SiO2"}

def contains_all_allowed_compounds(
    composition: str, allowed: set[str] = ALLOWED_COMPOUNDS
) -> bool:
  """Returns True if `composition` contains ALL target compounds/substrings."""
  if not isinstance(composition, str) or not composition.strip():
    return False
  return all(comp in composition for comp in allowed)


filtered_df = df[df["composition"].apply(contains_all_allowed_compounds)].copy()


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


print(f"\nKept rows: {len(filtered_df)}")
print(f"Removed rows: {len(df) - len(filtered_df)}")
print(filtered_df[["SID", "composition"]].head(10))





Kept rows: 0
Removed rows: 5173
Empty DataFrame
Columns: [SID, composition]
Index: []


Finding the "OR" compostion


In [264]:



import re
import cv2
import pandas as pd

df = pd.read_csv("updated_data.csv")
ALLOWED_COMPOUNDS = {"FeO3"}
#ALLOWED_COMPOUNDS = {"Ti", "Nb", "Bi", "Ba", "Na", "Pb", "K", "Mg"}


#"""Returns True if `composition` contains any target compound/substring."""
def contains_any_allowed_compound(
    composition: str, allowed: set[str] = ALLOWED_COMPOUNDS
) -> bool:
  if not isinstance(composition, str) or not composition.strip():
    return False
  return any(comp in composition for comp in allowed)


filtered_df = df[df["composition"].apply(contains_any_allowed_compound)].copy()

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print(f"\nKept rows: {len(filtered_df)}")
print(f"Removed rows: {len(df) - len(filtered_df)}")
print(filtered_df[["SID", "composition"]].head(10))


Kept rows: 519
Removed rows: 4654
        SID                                composition
367   51551        (BiFeO3)0.7(CaTiO3)0.15(SmFeO3)0.15
368   51551          (BiFeO3)0.6(CaTiO3)0.2(SmFeO3)0.2
369   51551        (BiFeO3)0.5(CaTiO3)0.25(SmFeO3)0.25
380   51558                                      YFeO3
462   51628       (BiFeO3)63.35(BaTiO3)34.71(MnO2)1.94
542   51767                   (BiFeO3)0.67(BaTiO3)0.33
543   51767  (BiFeO3)0.59(BaTiO3)0.33(Bi2LaTiNbO9)0.08
1156  52437                                     BiFeO3
1185  52461                                     BiFeO3
1186  52462                                     BiFeO3


Final Compostion


In [300]:
import pandas as pd

df = pd.read_csv("updated_data.csv")

#Ta2O5 or TaO2.5
#Nb2O5 or Nb02.5

ALLOWED_COMPOUNDS = {
    "MnO2", "CuO", "B2O3", "Al2O3", "MgO", "K2O", "Bi2O3", "Sm2O3", "Na2O",
    "ZnO", "SrO", "Fe2O3", "La2O3", "TiO2", "Eu2O3", "MnO", "Li2O", "ZrO2",
    "CaO", "Pr6O11", "Mn2O3", "NiO", "Ta2O5", "WO3", "CeO2", "Nd2O3", "CoO",
    "CoO1.333", "CoO4/3", "Cr2O3", "In2O3", "NbO2.5", "Sc2O3", "Tm2O3", "B2O5",
    "Co3O4", "Dy2O3", "GeO2", "VO4","BiScO3"
    "TaO2.5", "Er2O3", "Ho2O3", "SnO2", "Y2O3", "CdO","KNbO3"
    "Mn3O4", "Sb2O5", "Li2CO3", "K2CO3", "CaCO3", "MnCO3", "SrCO3", "BaCO3",
    "H3BO3", "MgCO3", "LiNO3", "Na2CO3","CaB4O7"
}


def contains_any_allowed_compound(
    composition: str, allowed: set[str] = ALLOWED_COMPOUNDS
) -> bool:
  if not isinstance(composition, str) or not composition.strip():
    return False
  return any(comp in composition for comp in allowed)


filtered_df = df[df["composition"].apply(contains_any_allowed_compound)].copy()

# Skip rows where `composition` is the same as the previous row
filtered_df = filtered_df[
    filtered_df["composition"].ne(filtered_df["composition"].shift(1))
].copy()

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print(f"\nKept rows: {len(filtered_df)}")
print(f"Removed rows: {len(df) - len(filtered_df)}")
print(filtered_df[["SID", "composition"]].head(10))


Kept rows: 655
Removed rows: 4518
       SID                                                                            composition
36   51359                                ((K0.2Sr0.4NbO3)3(PbMg0.33Nb0.67O3)0.7)88.65(Li2O)11.35
37   51359                                ((K0.2Sr0.4NbO3)4(PbMg0.33Nb0.67O3)0.6)86.74(Li2O)13.26
38   51359                                ((K0.2Sr0.4NbO3)5(PbMg0.33Nb0.67O3)0.5)84.91(Li2O)15.09
43   51359                               ((K0.2Sr0.4NbO3)5(PbMg0.33Nb0.67O3)0.5)75.75(LiNO3)24.25
45   51359                                ((K0.2Sr0.4NbO3)5(PbMg0.33Nb0.67O3)0.5)84.91(Li2O)15.09
54   51359  (((K0.2Sr0.4NbO3)4.5((PbMg0.33Nb0.67O3)0.77(PbFe0.5Nb0.5O3))0.23)0.55)93.27(Li2O)6.73
286  51492                                  (Sr0.61Ba0.39Nb2O6)95.96(BaO)1.33(B2O3)1.33(SiO2)1.33
287  51492                               (Sr0.61Ba0.39Nb2O6)92.17(BaO)2.584(B2O3)2.584(SiO2)2.584
288  51492                               (Sr0.61Ba0.39Nb2O6)88.59(BaO)3.765(B2O3)3.

In [305]:
import pandas as pd

df = pd.read_csv("updated_data.csv")
ALLOWED_COMPOUNDS = {"NbO2.5"}


def contains_any_allowed_compound(
    composition: str, allowed: set[str] = ALLOWED_COMPOUNDS
) -> bool:
  if not isinstance(composition, str) or not composition.strip():
    return False
  return any(comp in composition for comp in allowed)


filtered_df = df[df["composition"].apply(contains_any_allowed_compound)].copy()

# Skip rows where `composition` is the same as the previous row
filtered_df = filtered_df[
    filtered_df["composition"].ne(filtered_df["composition"].shift(1))
].copy()

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print(f"\nKept rows: {len(filtered_df)}")
print(f"Removed rows: {len(df) - len(filtered_df)}")
print(filtered_df[["SID", "composition"]].head(10))


Kept rows: 1
Removed rows: 5172
        SID                                               composition
4404  55072  BaTiO3((Bi0.5Na0.5)TiO3)0.05(NbO2.5)0.015(CoO1.333)0.005
